In [ ]:
!pip install Levenshtein spacy pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.4 MB/s eta 0:00:00


In [ ]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 22.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Reconhecimento de padrões via Regex

In [ ]:
import re

In [ ]:
text_corpus = """
Nota Oficial: A empresa TechInnovate (contato@techinnovate.com.br, https://techinnovate.dev)
comunicou que o especialista contratado sob o CPF 123.456.789-00 iniciará suas atividades
em 25/05/2026. Dúvidas com o RH em suporte-tecnico@tech.org ou acesse http://ajuda.tech.org/faq.
"""

In [ ]:
email_regex = r"[^\s]+@[a-zA-Z0-9\.]+\.[a-zA-Z]+"
url_regex = r"https?://(?:www\.)?[a-zA-Z0-9-_.]+\.[a-zA-Z]{2,}(?:/[^\s]*)?"
date_regex = r"\b\d{2}/\d{2}/\d{4}\b"
cpf_regex = r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b"

In [ ]:
print(re.findall(email_regex, text_corpus))
print(re.findall(url_regex, text_corpus))
print(re.findall(date_regex, text_corpus))
print(re.findall(cpf_regex, text_corpus))

['(contato@techinnovate.com.br', 'suporte-tecnico@tech.org']
['https://techinnovate.dev', 'http://ajuda.tech.org/faq.']
['25/05/2026']
['123.456.789-00']


# Correspodência Aproximada

In [ ]:
import Levenshtein

In [ ]:
import pandas as pd
input = "contato@techinovat.com"


email_database = [
    "contato@techinnovate.com.br",
    "suporte-tecnico@tech.org",
    "admin@innovate.dev"
]

df = pd.DataFrame({ 'email': email_database, 'input': [input] * 3} )
df


,email,input
0,contato@techinnovate.com.br,contato@techinovat.com
1,suporte-tecnico@tech.org,contato@techinovat.com
2,admin@innovate.dev,contato@techinovat.com


In [ ]:
df['distance'] = df.apply(lambda x: Levenshtein.distance(x['email'], x['input']), axis=1)
df

,email,input,distance
0,contato@techinnovate.com.br,contato@techinovat.com,5
1,suporte-tecnico@tech.org,contato@techinovat.com,16
2,admin@innovate.dev,contato@techinovat.com,16


In [ ]:
df.sort_values(by= 'distance').email.values[0]

'contato@techinnovate.com.br'

# Reconhecimento de Entidades Nomeadas

In [ ]:
import spacy


nlp = spacy.load("en_core_web_md")

news_article = """
US indicts former Cuban President Raúl Castro
"""

doc = nlp(news_article)

print(f"Entidades encontradas: {len(doc.ents)} ")

Entidades encontradas: 3 


In [ ]:
for ent in doc.ents:
    if ent.label_ in ["PERSON", "ORG", "GPE"]:
        print(f"Entidade: {ent.text:<12} | Tipo: {ent.label_:<8}")

Entidade: US           | Tipo: GPE     
Entidade: Raúl Castro  | Tipo: PERSON  


## Mais documentos

In [ ]:
news = [
    "Microsoft acquired GitHub.",
    "Google bought Kaggle.",
    "Apple opened offices in London.",
    "Microsoft rejected Google.",
    "Google challenged Microsoft.",
    "Apple acquired TechInnovate.",
    "Apple challenged Microsoft."
]

documents = [nlp(n) for n in news]
edges = [[
        entity.text for entity in doc.ents if entity.label_ in ['ORG']
    ]
    for doc in documents
]
edges = [e for e in edges if len(e) > 1]


In [ ]:
for doc in documents:
    verb = None
    for sent in doc.sents:
        for token in sent:
            if token.pos_ == 'VERB':
                verb = token.text
                break

    ent1, ent2 = None, None
    for ent in doc.ents:
        if ent.label_ == 'ORG':
            if ent1 is None:
                ent1 = ent.text
            else:
                ent2 = ent.text
                break
    print(f"{ent1}, {ent2}, {verb}")


Microsoft, GitHub, acquired
Google, Kaggle, bought
Apple, None, opened
Microsoft, Google, rejected
Google, Microsoft, challenged
Apple, TechInnovate, acquired
Apple, Microsoft, challenged


In [ ]:
df = pd.DataFrame(edges, columns=['source', 'target'])
df['weight'] = 1.

df.to_csv('edges.csv', index=False)

In [ ]:
from  itertools import chain
from collections import Counter

corps = list( chain(*edges) )
Counter(corps).most_common()


[('Microsoft', 4),
 ('Google', 3),
 ('Apple', 2),
 ('GitHub', 1),
 ('Kaggle', 1),
 ('TechInnovate', 1)]